# Docker Overview

A practical introduction to Docker, organized around three questions: **Why use it? What is it? How do I use it?**

> Goal: after this notebook, you should be able to run an application in a container, build your own image, and understand where Docker fits into a development workflow.

## 1. Why Docker?

Software often works on one computer but fails on another. The application may depend on a particular Python version, operating-system library, configuration setting, or environment variable. Installing those requirements manually can be slow and inconsistent.

Docker packages an application **together with the environment it needs**. Anyone who runs that package gets the same dependencies and startup instructions.

### Benefits

- **Consistency:** development, testing, and production can use the same environment.
- **Isolation:** projects can use different dependency versions without interfering with one another.
- **Reproducibility:** the image documents how the application environment is created.
- **Fast onboarding:** a new teammate can run a project with a few commands instead of a long installation guide.
- **Portability:** the same image can run on a laptop, a server, or a cloud platform that supports containers.

Docker is not automatically the best choice for every small script. It becomes especially valuable when an application has dependencies, several services, or needs to run reliably in more than one environment.

## 2. What is Docker?

Docker is a platform for building, sharing, and running **containers**. A container is an isolated process that runs an application with its required files and dependencies.

### The core concepts

| Term | Meaning | Simple analogy |
|---|---|---|
| **Image** | A read-only template containing an application and its environment. | A recipe or blueprint. |
| **Container** | A running instance of an image. | The meal made from the recipe. |
| **Dockerfile** | A text file with instructions for building an image. | The written recipe. |
| **Registry** | A service that stores and distributes images, such as Docker Hub. | A cookbook library. |
| **Volume** | Persistent storage managed by Docker. | A reusable data drive. |
| **Network** | A way for containers to communicate securely. | A private office network. |

### Containers compared with virtual machines

A virtual machine (VM) includes a full guest operating system. A container shares the host operating system's kernel, while isolating the application process and its filesystem. This usually makes containers smaller and faster to start than VMs.

Containers are isolation tools, not a complete security boundary by themselves. Use trusted images, keep them updated, and avoid placing secrets inside images.

### The Docker lifecycle

```text
Dockerfile  -->  docker build  -->  Image  -->  docker run  -->  Container
                                  |                         |
                                  +--> Registry <-----------+
```

1. Write a `Dockerfile` that describes the environment.
2. Build an image from the Dockerfile.
3. Run the image as one or more containers.
4. Optionally publish the image to a registry so that others can pull it.

## 3. How do I use Docker?

### Before you start

Install Docker Desktop (Windows or macOS) or Docker Engine (Linux), then confirm that Docker is running:

```bash
docker --version
docker run hello-world
```

The first command checks the installation. The second downloads a small test image, creates a container, and prints a confirmation message.

### Run an existing image

This starts the Nginx web server and exposes it on port `8080` on your computer:

```bash
docker run --name my-web-server -d -p 8080:80 nginx
```

Open `http://localhost:8080` in a browser. Docker maps **host port** `8080` to **container port** `80`.

Useful follow-up commands:

```bash
docker ps                         # List running containers
docker logs my-web-server         # View container output
docker stop my-web-server         # Stop the container
docker rm my-web-server           # Remove the stopped container
```

### Build your own image

Assume a Python web application has `app.py` and `requirements.txt`. Create a file named `Dockerfile` in the project root:

```dockerfile
# Start from a small Python environment
FROM python:3.12-slim

# Work inside this directory in the container
WORKDIR /app

# Copy dependency information first to make build caching useful
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the rest of the application
COPY . .

# Document the port used by the app
EXPOSE 8000

# Start the application
CMD ["python", "app.py"]
```

Build and run it from the same directory:

```bash
docker build -t my-python-app:1.0 .
docker run --rm -p 8000:8000 my-python-app:1.0
```

- `-t` gives the image a readable name and tag.
- The final `.` tells Docker to use the current directory as the build context.
- `--rm` removes the container automatically when it stops.

### Keep data when containers are replaced

A container's writable filesystem is temporary: deleting the container deletes its internal data. Use a **volume** for data that must survive, such as a database:

```bash
docker volume create postgres-data
docker run -d --name database \
  -e POSTGRES_PASSWORD=change-me \
  -v postgres-data:/var/lib/postgresql/data \
  postgres:16
```

For local development, you can also mount a project folder so edits on your computer appear immediately in the container:

```bash
docker run --rm -it -v "${PWD}:/app" -w /app python:3.12-slim bash
```

On PowerShell, replace `${PWD}` with `${PWD.Path}` if necessary.

### Run multiple services with Docker Compose

Applications often need more than one service, for example a web app and a database. Docker Compose describes them in one `compose.yaml` file:

```yaml
services:
  web:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - db

  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: change-me
    volumes:
      - postgres-data:/var/lib/postgresql/data

volumes:
  postgres-data:
```

Start the stack with `docker compose up --build`. Stop and remove the services with `docker compose down`. The named volume remains unless you add `--volumes`.

## Practical habits

- Use a `.dockerignore` file to exclude unnecessary files such as `.git`, virtual environments, and secrets from the build context.
- Pin important image versions (for example, `postgres:16`) instead of relying on `latest`.
- Keep images small and start from trusted, maintained base images.
- Put secrets in environment management or secret stores, not in a Dockerfile or image.
- Run one main responsibility per container when possible.
- Use `docker ps`, `docker logs`, and `docker exec -it <container> sh` to inspect a running container.

## Quick reference

| Task | Command |
|---|---|
| List images | `docker images` |
| List all containers | `docker ps -a` |
| Build an image | `docker build -t name:tag .` |
| Run an image | `docker run image:tag` |
| View logs | `docker logs <container>` |
| Open a shell | `docker exec -it <container> sh` |
| Stop all Compose services | `docker compose down` |

### Key takeaway

Docker makes an application's environment explicit and repeatable. You define it once in a Dockerfile, build an image, and run that image as a container wherever Docker is available.